In [ ]:
from OPET_control import OPETBus, OPET
from OPET_control.calibrate import run_calibration
from fluke5522a_calibrator import Calibrator
from serial import Serial
from datetime import datetime

# Set up instruments

## Set up and connect to the calibrator

In [ ]:
# Specify this port name as `/dev/tty.usbserial...` on *nix or as `COM99`
# on Windows
calibrator_port_name = 'COM5'
calibrator_port = Serial(
    calibrator_port_name,
    baudrate=9600,
    timeout=10,
    rtscts=True
)
calibrator = Calibrator(calibrator_port)

## Set up and connect to one OPET

In [ ]:
opet_adapter_serial_number = "AM00KJ6B"  # qrobots
opet_adapter_serial_number = "A10KNFU6"  # waveshare

# If you don't have `serial_by_serial`, you can just specify this port name
# as `/dev/tty.usbserial...` on *nix or as `COM99` on Windows
# opet_port_name = device_name(opet_adapter_serial_number)[0]\n",
opet_port_name = 'COM19'
opet_port = Serial(
    opet_port_name,
    baudrate=200000,
    timeout=1
)
opet_bus = OPETBus(opet_port)


### Select target OPET address

In [ ]:
opet_address = 15
opet = OPET(opet_bus, opet_address)

### Optional - Write name/serial number to OPET EEPROM (first calibration / commissioning only)

In [ ]:
opet_eepromsn = 2400210225 #Enter desired name string for target OPET
opet.write_eeprom(1,opet_eepromsn) #Write OPET name/serial number to EEPROM

## Set up a logging destination

In [ ]:
report_destination = f'/Users/bmcdanol/Documents/opet-calibration-results'

# Do calibration

## Voltage

### Connections for HC OPET
- `NORMAL LO` on calibrator to `PV Volt V+` on OPET
- `NORMAL HI` on calibrator to `PV Volt V–` on OPET
- `PV Volt V–` on OPET to `PV Volt S` on OPET
- Nothing else connected

In [ ]:
# Run the voltage calibration
result_voltage = run_calibration(
    calibrator, opet, 'voltage',
    update_calibration_constants=True,
    report_destination=report_destination
)

## Current (low ranges; HC and LC OPET)

### Connections for HC OPET
- `AUX LO` on calibrator to `BiasPWR GND` on OPET
- `AUX HI` on calibrator to `MOSFET S` on OPET
- Nothing else connected

In [ ]:
# Run the low ranges current calibration
result_current_low = run_calibration(
    calibrator, opet, 'current-low',
    validate_calibration_constants=False,
    update_calibration_constants=True,
    report_destination=report_destination
)

## Current (high ranges; HC OPET only)

### Connections for HC OPET
- `AUX LO` on calibrator to `BiasPWR GND` on OPET
- `AUX 20A` on calibrator to `MOSFET S` on OPET
- Nothing else connected

In [ ]:
# Run the high ranges current calibration
result_current_high = run_calibration(
    calibrator, opet, 'current-high',
    update_calibration_constants=False,
    report_destination=report_destination
)

In [ ]:
calibrator.current_output_posts = 'A20'